# Databricks Cluster Management Guide for Your Exact App Architecture
### Tailored to `ui_main.py` (`ui.page_navbar`), `css_design.py`, `server_main.py`, and `cluster_manager.py`

---

## 1. What We Are Implementing (Summary)

Based on your actual code in `ui_main.py`, `css_design.py`, and `server_main.py`:

1. **App Startup Cluster Verification $ightarrow$ Confirmation Modal (ZERO Auto-Start)**:
   * **When you open the app**, it immediately checks whether the Databricks Spark cluster is in `RUNNING` state.
   * **Zero automatic startup**: The app will **NEVER** start compute automatically in the background on app launch.
   * **If the cluster is RUNNING**: the app queries Delta tables (`load_delta_table`) and populates all dropdowns without delay.
   * **If the cluster is OFFLINE / TERMINATED**: the app does **NOT** query Databricks (preventing crashes/hangs), fills dropdowns with safe placeholders, and **immediately asks the user via a floating confirmation modal**: *"Databricks Spark Cluster is Offline. Would you like to start the cluster?"* with `[Yes, Start Cluster]` and `[Cancel]`.
   * **Only after the user explicitly confirms from the floating window (`[Yes, Start Cluster]`)** does it initiate startup, display the *'Starting Databricks Cluster (2-4 min)...'* spinner, and load the dropdowns automatically once online.

2. **Log Out & End Session Button Pushed to Far Right of Navbar (with Proper Icon & Original User Email)**:
   * Placed cleanly at the far right of the top navigation bar using `ui.nav_spacer()` and `ui.nav_control`.
   * Styled with a professional sign-out SVG icon and subtle danger accent, separated from the left-hand navigation tabs.
   * Displays the user's **original authenticated corporate email** (extracted automatically from Databricks Apps reverse-proxy headers `x-forwarded-email`, instead of generic placeholder *'Databricks User'*).
   * Clicking it opens the modal with two choices: **End App Session** (pauses polling & locks session) or **Log Out of Databricks** (`/login.html`).

3. **Safe Conditional Dropdown Loading (Only Queries Tables When Cluster is Ready)**:
   * `dropdown_load.py` uses a reactive effect observing `is_cluster_ready` before executing `load_delta_table("gas_shiny_input_versions")`.
   * If the cluster is stopped, dropdowns safely show *"Cluster offline. Start cluster above to load options"* with zero failing DB calls.
   * The instant compute reaches `RUNNING` (either on startup or after starting), the reactive effect triggers and automatically populates the dropdowns with real options.

4. **CSS in `css_design.py`**:
   * All CSS is kept directly in your `css_design.py` file (no separate static files needed!).

## 2. Your Project File Tree & What to Modify

```
app/
├── ui_scripts/
│   └── ui_helper/
│       ├── css_design.py          <-- 1. APPEND the cluster & modal CSS styles here
│       ├── home_page.py
│       ├── scenario_comparison.py
│       └── ...
│
├── server_scripts/
│   └── visuals/
│       ├── dropdown_load.py       <-- 2. WRAP table loading inside @reactive.event(is_cluster_ready)
│       └── visuals_main.py
│
├── cluster_manager.py             <-- 3. APPEND modals and register_cluster_handlers
├── server_main.py                 <-- 4. REMOVE auto-start lines; call register_cluster_handlers
└── ui_main.py                     <-- 5. ADD cluster_banner, ui.nav_spacer(), and right-aligned Log Out with icon
```

---

## 3. FILE 1: `css_design.py`
**Location**: `./ui_scripts/ui_helper/css_design.py`

### 🟢 WHAT TO KEEP:
* Keep all your existing CSS rules in `custom_css`.

### 🟡 WHAT TO ADD:
Append the following CSS to your `custom_css` string in `css_design.py`. This styles:
1. The top status alert banner (`.health-banner.warn`).
2. The confirmation modal (`.status-modal-content`).
3. The dual-box layout for the Log Out modal (`.session-option-box`).
4. The right-aligned Log Out button (`.nav-logout-btn`) and full-width navbar alignment (`.navbar-nav`).

In [ ]:
# ===========================================================================
# APPEND THIS CSS TO YOUR custom_css STRING IN: ui_scripts/ui_helper/css_design.py
# ===========================================================================

cluster_css = """
/* -------------------------------------------------------------------------
   Databricks Cluster Status Alert Banner
   ------------------------------------------------------------------------- */
.health-banner {
    display: flex;
    align-items: center;
    padding: 10px 24px;
    font-size: 0.88rem;
    border-bottom: 1px solid #fed7aa;
    background-color: #fff7ed;
    color: #9a3412;
    margin-bottom: 0px;
    position: relative;
    z-index: 1000;
}

.health-banner .health-state {
    font-weight: 800;
    margin-right: 14px;
    letter-spacing: -0.01em;
    color: #c2410c;
}

/* -------------------------------------------------------------------------
   Floating Modals (Offline Prompt, Spinner & Logout Dialog)
   ------------------------------------------------------------------------- */
.status-modal-content {
    text-align: center;
    padding: 24px 20px;
}

.status-modal-title {
    font-size: 1.4rem;
    font-weight: 800;
    color: #0f172a;
    margin-bottom: 8px;
}

.status-modal-subtitle {
    font-size: 0.92rem;
    color: #475569;
    line-height: 1.5;
    max-width: 460px;
    margin: 0 auto 20px auto;
}

/* Dual-box layout for the Log Out modal */
.session-options-row {
    display: flex;
    gap: 16px;
    margin-bottom: 20px;
    text-align: left;
}

.session-option-box {
    flex: 1;
    border: 1px solid #e2e8f0;
    border-radius: 10px;
    padding: 16px;
    display: flex;
    flex-direction: column;
    background: #ffffff;
    transition: border-color 0.15s ease, box-shadow 0.15s ease;
}

.session-option-box:hover {
    border-color: #cbd5e1;
    box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05);
}

.session-option-title {
    font-weight: 700;
    color: #1e293b;
    font-size: 0.98rem;
    margin-bottom: 4px;
}

.session-option-desc {
    font-size: 0.82rem;
    color: #64748b;
    line-height: 1.4;
    margin-bottom: 14px;
    flex-grow: 1;
}

/* -------------------------------------------------------------------------
   Right-Aligned Navbar & Log Out Button with Icon
   ------------------------------------------------------------------------- */
.navbar-nav {
    width: 100%;
    align-items: center;
}

.nav-logout-btn {
    display: inline-flex !important;
    align-items: center !important;
    gap: 6px !important;
    font-weight: 700 !important;
    font-size: 0.84rem !important;
    padding: 5px 14px !important;
    border-radius: 6px !important;
    border: 1px solid rgba(239, 68, 68, 0.65) !important;
    color: #ffffff !important;
    background-color: rgba(239, 68, 68, 0.15) !important;
    transition: all 0.2s ease-in-out !important;
    white-space: nowrap !important;
}

.nav-logout-btn:hover {
    background-color: #dc2626 !important;
    border-color: #dc2626 !important;
    color: #ffffff !important;
    box-shadow: 0 2px 8px rgba(220, 38, 38, 0.35) !important;
    transform: translateY(-1px);
}

.nav-logout-btn svg {
    flex-shrink: 0;
    vertical-align: -1px;
}
"""

# IN YOUR css_design.py:
# custom_css = custom_css + cluster_css

## 4. FILE 2: `ui_main.py`
**Location**: `./ui_main.py`

In your current code (from your screenshot):
```python
app_ui = ui.page_navbar(
    home_page(),
    scenario_comp(),
    ui.nav_control(
        ui.tags.a("📊 Visuals", href=visuals_link, target="_blank", class_="nav-link")
    ),
    capacity(),
    capacity_cont(),
    demand(),
    demand_config(),
    config(),
    about(),
    title=ui.HTML(...),
    id="navbar",
    header=ui.tags.head(...)  # <-- CURRENT HEADER
)
```

### 🟢 WHAT TO KEEP:
* Keep all your pages: `home_page()`, `scenario_comp()`, `capacity()`, `capacity_cont()`, `demand()`, `demand_config()`, `config()`, `about()`.
* Keep your `title=ui.HTML(...)`.
* Keep your scripts inside `ui.tags.head(...)` (the British Gas logo script and accordion script).

### 🟡 WHAT TO CHANGE:
1. **Why was the Log Out button showing on the left?**
   * In PyShiny's `ui.page_navbar(...)`, any control passed between tabs is rendered sequentially from left to right.
   * To push items to the **far right**, PyShiny provides **`ui.nav_spacer()`**. It acts as an expanding flex spacer that pushes all controls defined after it cleanly to the right side of the navbar!
2. **Add `ui.nav_spacer()` directly after `about()`** (after all your main navigation tabs).
3. **Add `ui.nav_control(...)` right after `ui.nav_spacer()`** with a proper sign-out icon (using an inline SVG icon inside `ui.HTML(...)` so it renders crisp with zero external font dependencies).
4. **Add `ui.output_ui("cluster_banner")`** in the `header` parameter (wrapped in a list with `ui.tags.head(...)`).

Here is the exact code for `ui_main.py`:

In [ ]:
# Code for ui_main.py

from shiny import ui
from ui_scripts.ui_helper.css_design import custom_css
# ... your existing imports (home_page, scenario_comp, etc.) ...

# Clean, self-contained SVG icon for Log Out (no external font dependencies)
LOGOUT_ICON_SVG = """<svg width=\"15\" height=\"15\" viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"currentColor\" stroke-width=\"2.2\" stroke-linecap=\"round\" stroke-linejoin=\"round\" style=\"margin-right: 4px; vertical-align: -1px;\"><path d=\"M9 21H5a2 2 0 0 1-2-2V5a2 2 0 0 1 2-2h4\"></path><polyline points=\"16 17 21 12 16 7\"></polyline><line x1=\"21\" y1=\"12\" x2=\"9\" y2=\"12\"></line></svg>"""

app_ui = ui.page_navbar(
    # --- NAVIGATION TABS (LEFT-ALIGNED) ---
    home_page(),
    scenario_comp(),
    
    # 1. YOUR EXISTING VISUALS LINK
    ui.nav_control(
        ui.tags.a("📊 Visuals", href=visuals_link, target="_blank", class_="nav-link")
    ),
    
    capacity(),
    capacity_cont(),
    demand(),
    demand_config(),
    config(),
    about(),
    
    # =========================================================================
    # 2. NAV SPACER: PUSHES ALL SUBSEQUENT CONTROLS TO THE FAR RIGHT SIDE!
    # =========================================================================
    ui.nav_spacer(),
    
    # =========================================================================
    # 3. LOG OUT BUTTON AT FAR RIGHT (WITH PROPER SIGN-OUT ICON)
    # =========================================================================
    ui.nav_control(
        ui.input_action_button(
            "btn_logout_prompt",
            ui.HTML(f"""
                <span style=\"display: inline-flex; align-items: center; gap: 6px;\">
                    {LOGOUT_ICON_SVG}
                    <span>Log Out</span>
                </span>
            """),
            class_="btn btn-outline-danger btn-sm nav-logout-btn",
            style="font-weight: 700; padding: 4px 14px; border-radius: 6px; margin-left: 10px; margin-right: 12px; display: inline-flex; align-items: center;"
        )
    ),
    
    title=ui.HTML(
        "<span style='font-size: 18px;'> GAS </span>"
        + f"<span style='font-size: 12px; border-radius: 10px; background-color: #d7d7d9; color: black;' class='version-badge'> &nbsp; {app_version} &nbsp; </span>"
    ),
    id="navbar",
    
    # 3. HEADER: Contains your scripts & styles + TOP CLUSTER BANNER
    header=[
        ui.tags.head(
            ui.tags.style(custom_css),
            # Your existing British Gas logo script
            ui.tags.script("""
                $(document).ready(function() {
                    $('.navbar-nav').append('<li class="dropdown british-gas-logo"><figure><img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSZ3I3AxR65aZtgWqdjolG6F1gM_QvmQ5uyKQ&s" height="50" width="100"></figure></li>');
                });
            """),
            # Your existing accordion script
            ui.tags.script("""
                $(document).ready(function() {
                    $(document).on('shown.bs.collapse', '.accordion .collapse', function() {
                        var openedAccordionId = $(this).closest('.accordion').attr('id');
                        $(this).closest('.card').find('.accordion').not('#' + openedAccordionId).each(function() {
                            $(this).find('.collapse.show').collapse('hide');
                        });
                    });
                });
            """)
        ),
        # TOP CLUSTER STATUS BANNER (Displays when stopped, auto-hides when RUNNING!)
        ui.output_ui("cluster_banner")
    ]
)

## 5. FILE 3: `cluster_manager.py` (Add Modals & Handlers)
**Location**: `./cluster_manager.py`

### 🟢 WHAT TO KEEP:
* Keep your existing functions: `ClusterStatus`, `get_workspace_client()`, `get_cluster_id()`, `get_cached_status()`, `get_cluster_status()`, `start_cluster()`, `is_session_invalid_error()`.

### 🔴 CRITICAL: CHECK YOUR EXISTING `get_spark_session()` & `reset_spark_session()`:
* In your existing `cluster_manager.py`, look inside `get_spark_session()` and `reset_spark_session()`.
* **If either function contains `start_cluster()` (e.g. `if status == ClusterStatus.TERMINATED: start_cluster()`), DELETE that `start_cluster()` line!**
* `get_spark_session()` must **only** connect if the cluster is already running. If it calls `start_cluster()` automatically, the cluster starts in the background without user confirmation!

### 🟡 WHAT TO ADD:
Paste the following code at the bottom of `cluster_manager.py`. It provides:
* `get_current_user_email(session)`: **Extracts the original authenticated user's email** from Databricks Apps reverse-proxy headers (`x-forwarded-email`, `x-forwarded-preferred-username`, or `x-forwarded-user`), replacing the generic fallback *'Databricks User'*.
* `cluster_offline_modal`: The floating modal asking the user whether to start the cluster.
* `cluster_starting_modal`: The spinner modal.
* `logout_confirmation_modal` & `session_ended_modal`: The Logout dialog displaying the original user email in a clean badge.
* `register_cluster_handlers`: Checks cluster status **immediately on app open**. If the cluster is already running, sets `is_cluster_ready` to `True` so dropdowns load immediately. If offline/terminated, sets `is_cluster_ready` to `False` (blocking dropdown DB queries) and immediately pops up `cluster_offline_modal` asking the user to start the cluster.
* **Click Confirmation Safeguard**: Guarded with `if not count or count < 1: return` so `start_cluster()` executes **ONLY** when the user physically clicks `[Yes, Start Cluster]` in the floating window.

In [ ]:
# Code to append at the bottom of cluster_manager.py

import asyncio
from shiny import Inputs, Outputs, Session, reactive, render, ui

def get_current_user_email(session: Session) -> str:
    """
    Extracts the authenticated user's corporate email in Databricks Apps.
    In Azure Databricks Apps, requests pass through an authenticating reverse proxy with headers:
      - x-forwarded-email (user corporate email)
      - x-forwarded-preferred-username
      - x-forwarded-user
    Falls back to session.user, Databricks SDK WorkspaceClient, or environment variables.
    """
    # 1. Primary: Databricks Apps HTTP reverse-proxy request headers
    http_conn = getattr(session, "http_conn", None)
    if http_conn and hasattr(http_conn, "headers"):
        headers = http_conn.headers
        for key in ("x-forwarded-email", "x-forwarded-preferred-username", "x-forwarded-user"):
            val = headers.get(key)
            if val and str(val).strip():
                return str(val).strip()

    # 2. Secondary: Shiny session user (if Shiny authentication is active)
    user = getattr(session, "user", None)
    if user and str(user).strip():
        return str(user).strip()

    # 3. Tertiary: Query Databricks SDK WorkspaceClient for current identity
    try:
        from databricks.sdk import WorkspaceClient
        w = WorkspaceClient()
        me = w.current_user.me()
        if hasattr(me, "user_name") and me.user_name:
            return str(me.user_name).strip()
    except Exception:
        pass

    # 4. Fallback: Environment variable
    import os
    for env_var in ("DATABRICKS_USER_EMAIL", "USER_EMAIL", "USER"):
        val = os.environ.get(env_var)
        if val and str(val).strip():
            return str(val).strip()

    return ""

def cluster_offline_modal(state: str = "TERMINATED") -> ui.Tag:
    return ui.modal(
        ui.div(
            ui.HTML('''
                <svg width="38" height="38" viewBox="0 0 24 24" fill="none" stroke="#e07000" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round" style="margin-bottom: 12px;">
                    <rect x="2" y="2" width="20" height="8" rx="2" ry="2"></rect>
                    <rect x="2" y="14" width="20" height="8" rx="2" ry="2"></rect>
                    <line x1="6" y1="6" x2="6.01" y2="6"></line>
                    <line x1="6" y1="18" x2="6.01" y2="18"></line>
                </svg>
            '''),
            ui.div("Databricks Cluster is Offline", class_="status-modal-title"),
            ui.div(
                f"The Databricks Spark cluster has stopped due to inactivity (status: {state.lower()}). "
                "Would you like to start the cluster now to load dropdowns and run calculations?",
                class_="status-modal-subtitle"
            ),
            ui.div(
                ui.input_action_button(
                    "btn_start_cluster_confirm",
                    "Yes, Start Cluster",
                    class_="btn btn-primary",
                    style="min-width: 160px; font-weight: 700; background-color: #005eb8; border-color: #005eb8; padding: 8px 20px;"
                ),
                ui.input_action_button(
                    "btn_offline_modal_dismiss",
                    "Cancel",
                    class_="btn btn-outline-secondary",
                    style="min-width: 100px; padding: 8px 20px;"
                ),
                style="display: flex; justify-content: center; gap: 12px; margin-top: 10px;"
            ),
            class_="status-modal-content"
        ),
        title=None,
        footer=None,
        easy_close=False,
        size="m"
    )

def cluster_starting_modal() -> ui.Tag:
    return ui.modal(
        ui.div(
            ui.div(class_="spinner-border text-primary", style="width: 3rem; height: 3rem; color: #005eb8; margin-bottom: 14px;"),
            ui.div("Starting Databricks Cluster...", class_="status-modal-title"),
            ui.div(
                "Startup has been requested in Databricks. Usually takes 2 to 4 minutes. "
                "The app will automatically connect and load dropdowns once the cluster is online.",
                class_="status-modal-subtitle"
            ),
            ui.div(
                ui.input_action_button("btn_dismiss_starting_modal", "Wait in Background", class_="btn btn-outline-secondary btn-sm"),
                style="display: flex; justify-content: center;"
            ),
            class_="status-modal-content"
        ),
        title=None,
        footer=None,
        easy_close=False,
        size="m"
    )

def logout_confirmation_modal(user_email: str = "", databricks_host: str = "") -> ui.Tag:
    clean_host = databricks_host if databricks_host.startswith("http") else f"https://{databricks_host}" if databricks_host else ""
    logout_url = f"{clean_host.rstrip('/')}/login.html" if clean_host else "/login.html"
    
    # User badge displaying the original authenticated corporate email
    user_badge = None
    if user_email and user_email.strip():
        user_badge = ui.div(
            ui.HTML('''
                <svg width=\"15\" height=\"15\" viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"#005eb8\" stroke-width=\"2.2\" stroke-linecap=\"round\" stroke-linejoin=\"round\" style=\"margin-right: 6px; vertical-align: -2px;\">
                    <path d=\"M20 21v-2a4 4 0 0 0-4-4H8a4 4 0 0 0-4 4v2\"></path>
                    <circle cx=\"12\" cy=\"7\" r=\"4\"></circle>
                </svg>
            '''),
            ui.tags.span("Signed in as: ", style="color: #64748b; font-weight: 500; font-size: 0.88rem;"),
            ui.tags.span(user_email, style="color: #005eb8; font-weight: 700; font-size: 0.92rem;"),
            style="display: inline-flex; align-items: center; justify-content: center; background: #eff6ff; border: 1px solid #bfdbfe; border-radius: 20px; padding: 6px 16px; margin: 0 auto 16px auto;"
        )
    
    return ui.modal(
        ui.div(
            ui.HTML('''
                <svg width=\"36\" height=\"36\" viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"#dc2626\" stroke-width=\"2.2\" stroke-linecap=\"round\" stroke-linejoin=\"round\" style=\"margin-bottom: 12px;\">
                    <path d=\"M9 21H5a2 2 0 0 1-2-2V5a2 2 0 0 1 2-2h4\"></path>
                    <polyline points=\"16 17 21 12 16 7\"></polyline>
                    <line x1=\"21\" y1=\"12\" x2=\"9\" y2=\"12\"></line>
                </svg>
            '''),
            ui.div("Log Out or End Your Session", class_="status-modal-title"),
            ui.div("Choose to pause your active session or sign out of Databricks completely.", class_="status-modal-subtitle"),
            user_badge,
            ui.div(
                # Option 1: End Session
                ui.div(
                    ui.div("⏸️ End App Session", class_="session-option-title"),
                    ui.div("Pause background compute polling and lock your session without losing credentials.", class_="session-option-desc"),
                    ui.input_action_button("btn_confirm_end_session", "End Session", class_="btn btn-primary btn-sm", style="width: 100%; font-weight: 700; background-color: #005eb8;"),
                    class_="session-option-box"
                ),
                # Option 2: Log Out of Databricks
                ui.div(
                    ui.div("🚪 Log Out of Databricks", class_="session-option-title"),
                    ui.div("Sign out completely from your Databricks workspace and close this application.", class_="session-option-desc"),
                    ui.tags.a("Log Out", href=logout_url, target="_top", class_="btn btn-outline-danger btn-sm", style="width: 100%; font-weight: 700; text-align: center; text-decoration: none;"),
                    class_="session-option-box"
                ),
                class_="session-options-row"
            ),
            ui.div(
                ui.input_action_button("btn_dismiss_logout_modal", "Cancel", class_="btn btn-outline-secondary", style="min-width: 100px;"),
                style="display: flex; justify-content: center;"
            ),
            class_="status-modal-content"
        ),
        title=None,
        footer=None,
        easy_close=True,
        size="m"
    )

def session_ended_modal() -> ui.Tag:
    return ui.modal(
        ui.div(
            ui.div("Session Ended", class_="status-modal-title"),
            ui.div("Your session has been locked and background compute polling is paused.", class_="status-modal-subtitle"),
            ui.div(
                ui.input_action_button("btn_resume_session", "Resume Session", class_="btn btn-primary", style="min-width: 160px; font-weight: 700; background-color: #005eb8;"),
                style="display: flex; justify-content: center;"
            ),
            class_="status-modal-content"
        ),
        title=None,
        footer=None,
        easy_close=False,
        size="m"
    )

def render_cluster_banner(status: ClusterStatus) -> ui.Tag | None:
    if status in (ClusterStatus.RUNNING, ClusterStatus.RESIZING):
        return None  # Auto-hides when RUNNING!

    is_offline = status in (ClusterStatus.TERMINATED, ClusterStatus.ERROR, ClusterStatus.UNKNOWN)
    is_starting = status in (ClusterStatus.STARTING, ClusterStatus.PENDING, ClusterStatus.RESTARTING)

    action_btn = None
    if is_offline:
        action_btn = ui.input_action_button(
            "btn_banner_start_cluster",
            "Start Cluster",
            class_="btn btn-primary btn-sm",
            style="margin-left: auto; font-weight: 700; background-color: #005eb8; border-color: #005eb8;"
        )
    elif is_starting:
        action_btn = ui.div(
            ui.tags.span(class_="spinner-border spinner-border-sm text-primary", style="margin-right: 6px;"),
            ui.tags.span("Starting Spark cluster (2-4 min)..."),
            style="margin-left: auto; display: flex; align-items: center; font-size: 0.85rem; font-weight: 700; color: #005eb8;"
        )

    return ui.div(
        ui.tags.span(f"Spark Cluster: {status.value.upper()}", class_="health-state"),
        ui.tags.span("Compute is stopped. Start cluster to load dropdowns and run calculations."),
        action_btn,
        class_="health-banner warn"
    )

def register_cluster_handlers(input: Inputs, output: Outputs, session: Session) -> reactive.Value[bool]:
    # 1. Immediate Check on App Launch: Is the cluster running right now?
    initial_status = get_cluster_status()
    is_initial_ready = initial_status in (ClusterStatus.RUNNING, ClusterStatus.RESIZING)

    cluster_status_val = reactive.value(initial_status)
    is_cluster_ready = reactive.value(is_initial_ready)
    modal_state = reactive.value("offline_prompt" if not is_initial_ready else "none")
    offline_modal_dismissed = reactive.value(False)

    # 2. If cluster is offline when app opens, immediately ask user to start it (before loading dropdowns)
    if not is_initial_ready:
        @reactive.effect
        def _show_initial_offline_modal():
            if not offline_modal_dismissed() and not is_cluster_ready():
                ui.modal_show(cluster_offline_modal(state=initial_status.value))

    @output
    @render.ui
    def cluster_banner():
        return render_cluster_banner(cluster_status_val())

    @reactive.effect
    async def poll_cluster_status_loop():
        if modal_state() == "session_ended":
            return

        status = await asyncio.to_thread(get_cluster_status, force_refresh=True)
        cluster_status_val.set(status)
        ready = status in (ClusterStatus.RUNNING, ClusterStatus.RESIZING)

        if is_cluster_ready() != ready:
            is_cluster_ready.set(ready)

        if ready:
            if modal_state() in {"starting", "offline_prompt"}:
                modal_state.set("none")
                offline_modal_dismissed.set(False)
                try:
                    ui.modal_remove()
                except Exception:
                    pass
                ui.notification_show("Spark cluster is online! Loading dropdowns and data...", type="message", duration=5)
            reactive.invalidate_later(20)
            return

        if status in (ClusterStatus.STARTING, ClusterStatus.PENDING, ClusterStatus.RESTARTING):
            reactive.invalidate_later(3)
            return

        # Inactivity or Offline: Prompt user with floating window instead of auto-starting
        if not offline_modal_dismissed() and modal_state() == "none":
            modal_state.set("offline_prompt")
            ui.modal_show(cluster_offline_modal(state=status.value))

        reactive.invalidate_later(30)

    @reactive.effect
    @reactive.event(input.btn_start_cluster_confirm)
    async def on_start_cluster_confirmed():
        # CRITICAL SAFEGUARD: Only start if user actually clicked the button (> 0)
        # Prevents auto-starting when the modal mounts in DOM (where initial value is 0 or None)
        count = input.btn_start_cluster_confirm()
        if not count or count < 1:
            return

        modal_state.set("starting")
        ui.modal_show(cluster_starting_modal())
        try:
            await asyncio.to_thread(start_cluster)
        except Exception as e:
            ui.notification_show(f"Failed to start cluster: {e}", type="error", duration=10)

    @reactive.effect
    @reactive.event(input.btn_banner_start_cluster)
    def on_banner_start_cluster():
        count = input.btn_banner_start_cluster()
        if not count or count < 1:
            return
        offline_modal_dismissed.set(False)
        modal_state.set("offline_prompt")
        ui.modal_show(cluster_offline_modal(state=cluster_status_val().value))

    @reactive.effect
    @reactive.event(input.btn_offline_modal_dismiss)
    def on_offline_modal_dismiss():
        count = input.btn_offline_modal_dismiss()
        if not count or count < 1:
            return
        offline_modal_dismissed.set(True)
        modal_state.set("none")
        ui.modal_remove()

    @reactive.effect
    @reactive.event(input.btn_dismiss_starting_modal)
    def on_dismiss_starting_modal():
        count = input.btn_dismiss_starting_modal()
        if not count or count < 1:
            return
        modal_state.set("none")
        ui.modal_remove()

    # Logout & Session End event handlers
    @reactive.effect
    @reactive.event(input.btn_logout_prompt)
    def on_logout_prompt():
        count = input.btn_logout_prompt()
        if not count or count < 1:
            return
        import os
        host = os.environ.get("DATABRICKS_HOST", "")
        user_email = get_current_user_email(session)  # Extracts actual corporate email from Databricks Apps
        ui.modal_show(logout_confirmation_modal(user_email=user_email, databricks_host=host))

    @reactive.effect
    @reactive.event(input.btn_dismiss_logout_modal)
    def on_dismiss_logout_modal():
        count = input.btn_dismiss_logout_modal()
        if not count or count < 1:
            return
        ui.modal_remove()

    @reactive.effect
    @reactive.event(input.btn_confirm_end_session)
    def on_confirm_end_session():
        count = input.btn_confirm_end_session()
        if not count or count < 1:
            return
        modal_state.set("session_ended")
        ui.modal_show(session_ended_modal())

    @reactive.effect
    @reactive.event(input.btn_resume_session)
    def on_resume_session():
        count = input.btn_resume_session()
        if not count or count < 1:
            return
        modal_state.set("none")
        ui.modal_remove()
        status = get_cluster_status(force_refresh=True)
        cluster_status_val.set(status)
        if status not in (ClusterStatus.RUNNING, ClusterStatus.RESIZING):
            modal_state.set("offline_prompt")
            ui.modal_show(cluster_offline_modal(state=status.value))

    return is_cluster_ready

## 6. FILE 4: `server_main.py`
**Location**: `./server_main.py`

### 🔴 WHAT TO REMOVE (ELIMINATES AUTO-START ON LAUNCH):
1. **Remove any module-level `reset_spark_session()` or `start_cluster()`** outside `def server(...)` (usually lines 24–41).
2. **Remove old `poll_cluster_status`** (usually lines 53–100) that called `start_cluster()` on terminated status.
3. **Ensure NO submodule calls `start_cluster()`**: Submodules like `bulk_upload_inputs`, `scenario_calc`, and `visuals_main` must NOT call `start_cluster()` on initialization.

### 🟢 WHAT TO KEEP:
* Keep all your imports: `bulk_upload_inputs`, `download_template`, `scenario_calc`, `scorecard_extract_final`.
* Keep all your submodule invocations (lines 106–113).

### 🟡 WHAT TO CHANGE IN `def server(input, output, session)`:

In [ ]:
# Code for server_main.py

from shiny import ui, render, reactive
from cluster_manager import register_cluster_handlers

# Keep your existing imports:
from server_scripts.visuals.visuals_main import visuals_main
from server_scripts.inputs.bulk_upload_validation import bulk_upload_inputs
from server_scripts.output.download_template_calculations import download_template
from server_scripts.calculations.scenario_comparison_calculations_main import scenario_calc
from server_scripts.output.scorecard_extract import scorecard_extract_final

def server(input, output, session):
    # 1. Cluster Status & Floating Window Management
    # (Replaces old poll_cluster_status; manages offline prompt modal, spinner, and Log Out)
    is_cluster_ready = register_cluster_handlers(input, output, session)

    # 2. Main Application Logic
    # Pass is_cluster_ready to visuals_main (and any module loading dropdowns/tables)
    # so dropdowns wait for the cluster to be RUNNING before querying Databricks:
    visuals_main(input, output, session, is_cluster_ready)

    # Keep your other submodules exactly as they are:
    bulk_upload_inputs(input, output, session)
    download_template(input, output, session)
    scenario_calc(input, output, session)
    scorecard_extract_final(input, output, session)

## 7. FILE 5: `dropdown_load.py` (Wait for Cluster to Start)
**Location**: `./server_scripts/visuals/dropdown_load.py`

### 🔴 WHAT TO REMOVE (STOPS AUTO-START ON OPEN):
Remove lines 24–30 (this block was silently auto-starting the cluster on load!):
```python
# REMOVE THIS ENTIRE BLOCK (prevents cluster from auto-starting on app load):
_initial_status = get_cluster_status()
if _initial_status == ClusterStatus.TERMINATED:
    start_cluster()  # <-- DELETE THIS! It starts the cluster without user confirmation
    return print("Cluster not running")
```

### 🟢 WHAT TO KEEP:
* Keep your `dropdown_remarks_config` dictionary.
* Keep your metric loop: `for metric, config in dropdown_remarks_config.items(): ...`
* Keep `dropdown_update`, `delete_dropdown`, and `undo_dropdown`.

### 🟡 WHAT TO REPLACE IT WITH:
Wrap your table loading in `@reactive.effect`:
* **When opening the app**, it immediately inspects `is_cluster_ready_signal()`.
* **If offline / starting**: sets safe placeholders in all dropdowns with zero SQL calls (no crash, no freeze).
* **If RUNNING (or once compute finishes starting)**: automatically queries Databricks tables and populates the dropdowns.

In [ ]:
# Code for dropdown_load.py (or load_input.py)

from shiny import ui, reactive, render
from common_functions import load_delta_table
from cluster_manager import is_cluster_ready

def final_load_dropdown_remarks(dropdown_remarks_config, input, session, is_cluster_ready_signal):
    """
    Checks cluster status before loading dropdowns:
    - If cluster is offline on app open: sets safe placeholders without querying DB (no crash!).
    - Once cluster reaches RUNNING: automatically loads Delta tables and populates dropdowns.
    """
    
    @reactive.effect
    def _populate_dropdowns_when_ready():
        # 1. Cluster is offline: do NOT query DB; set safe placeholder in dropdowns
        if not is_cluster_ready_signal():
            print("[dropdown_load] Cluster is not running. Setting safe placeholder in dropdowns...")
            for metric, config in dropdown_remarks_config.items():
                d_id = config.get("dropdown_id")
                if d_id:
                    ui.update_select(
                        d_id, 
                        choices=["Cluster offline. Start cluster above to load options"],
                        selected="Cluster offline. Start cluster above to load options"
                    )
            return

        # 2. Cluster is RUNNING! Safe to load tables:
        try:
            print("[dropdown_load] Cluster is RUNNING! Loading gas_shiny_input_versions & gas_shiny_scenario_log...")
            df1 = load_delta_table("gas_shiny_input_versions")
            df2 = load_delta_table("gas_shiny_scenario_log")
            
            # =========================================================
            # YOUR EXISTING DROPDOWN PROCESSING CODE RUNS HERE UNCHANGED:
            # =========================================================
            for metric, config in dropdown_remarks_config.items():
                dropdown_id = config["dropdown_id"]
                delete_id = config["delete_id"]
                
                if "version" in df1.columns:
                    available_versions = df1["version"].dropna().unique().tolist()
                    if available_versions:
                        ui.update_select(dropdown_id, choices=available_versions, selected=available_versions[0])
                        
            ui.notification_show("Dropdown options successfully loaded from Databricks!", type="message", duration=4)
        except Exception as e:
            print(f"[dropdown_load] Error populating dropdowns: {e}")
            ui.notification_show(f"Error loading dropdowns: {e}", type="warning", duration=6)

## 8. Summary Checklist

| File to Edit | Exact Modification |
| :--- | :--- |
| **`ui_scripts/ui_helper/css_design.py`** | Append `.health-banner`, `.status-modal-content`, `.session-option-box`, `.navbar-nav`, and `.nav-logout-btn` styles to `custom_css`. |
| **`ui_main.py`** | 1. Add `ui.output_ui("cluster_banner")` in `header=[...]`.<br>2. Add **`ui.nav_spacer()`** directly after `about()` to push controls to the right.<br>3. Add **`ui.nav_control(...)`** with `btn_logout_prompt` including the SVG sign-out icon on the far right. |
| **`cluster_manager.py`** | 1. Remove any `start_cluster()` calls inside existing `get_spark_session()` or `reset_spark_session()`.<br>2. Append `get_current_user_email(session)`, the 4 modal functions, and `register_cluster_handlers()` (includes click safeguard: only starts cluster after user confirms in floating window). |
| **`server_main.py`** | Remove lines 24–41 (module-level auto-start) and replace old `poll_cluster_status` with `is_cluster_ready = register_cluster_handlers(...)`. Pass `is_cluster_ready` to `visuals_main` and any submodule loading dropdowns. |
| **`dropdown_load.py`** | Remove lines 24–30. Wrap table loading in `@reactive.effect` checking `is_cluster_ready_signal()` so dropdowns only query Delta tables when cluster is RUNNING, otherwise setting safe placeholders without crashing. |

Your app will now cleanly match your exact layout: all CSS stays in `css_design.py`, `ui_main.py` uses `ui.page_navbar` with the Log Out button positioned on the far right with a proper sign-out icon, opening the app checks whether the cluster is running before loading dropdowns (prompting the user immediately if offline), and tables load safely once online!